In [1]:
from datasets import load_dataset
# 1. Dataset de Salud Mental (SomosNLP)
# Este ya viene en español nativo
mental_health = load_dataset("somosnlp/mental-health-corpus", split='train')

# 2. Dataset de Sentimientos (Pysentimiento)
# Útil para detectar el "tono" negativo/positivo
tweets_sentimiento = load_dataset("pysentimiento/spanish-tweets", split='train')

DatasetNotFoundError: Dataset 'somosnlp/mental-health-corpus' doesn't exist on the Hub or cannot be accessed.

### Limpieza y traducción a caracteres del español

In [1]:
import pandas as pd

archivo = 'combined_data_clean (1) (2).csv'

# 1. SOLUCIÓN AL ENCODING: Usamos 'latin-1' para leer los acentos de Excel/Windows
try:
    df = pd.read_csv(archivo, encoding='latin-1')
except:
    df = pd.read_csv(archivo, encoding='cp1252') # Alternativa para Windows

# 2. FILTRO DE CALIDAD: Solo usamos lo que NO necesita retraducción
# Esto elimina automáticamente la "traducción pésima"
df_bueno = df[df['needs_retranslate'].astype(str).str.strip().isin(['False', 'false', 'FALSE'])]

# 3. BALANCEO INTELIGENTE (En lugar de solo 100, usemos 500-1000)
# 100 es muy poco para RoBERTuito. 500-1000 es el punto ideal para 8GB de RAM.
n_muestras = 500 
categorias = ['Anxiety', 'Depression', 'Suicidal', 'Stress', 'Normal']

dataset_final = pd.DataFrame()

for cat in categorias:
    subset = df_bueno[df_bueno['status'] == cat]
    if len(subset) >= n_muestras:
        dataset_final = pd.concat([dataset_final, subset.sample(n_muestras, random_state=42)])
    else:
        # Si la categoría es pequeña (como Stress), tomamos todas las que haya
        dataset_final = pd.concat([dataset_final, subset])

# 4. LIMPIEZA FINAL Y EXPORTACIÓN
dataset_final = dataset_final[['statement_es', 'status']]
dataset_final.columns = ['text', 'label']

# Guardamos en UTF-8 para que RoBERTuito lo lea perfecto
dataset_final.to_csv('dataset_fase2_listo.csv', index=False, encoding='utf-8')

print(f"¡Listo! Dataset creado con {len(dataset_final)} filas sin caracteres raros.")

¡Listo! Dataset creado con 2500 filas sin caracteres raros.


In [2]:
import pandas as pd
import ftfy 

# 1. Leer el archivo que tiene los caracteres "raros"
# Probablemente se guardó mal, así que intentamos leerlo ignorando errores o con latin-1
df = pd.read_csv('dataset_fase2_listo.csv', encoding='utf-8')

# 2. Función mágica para arreglar acentos y signos (¿, ¡, á, é...)
def arreglar_texto(texto):
    if pd.isna(texto):
        return texto
    # ftfy arregla automáticamente "Ã³" a "ó", "Â¿" a "¿", etc.
    return ftfy.fix_text(texto)

# Aplicar a la columna de texto
df['text'] = df['text'].apply(arreglar_texto)

# 3. GUARDAR CORRECTAMENTE
# El truco es 'utf-8-sig' (especialmente para que Excel y VS Code lo lean perfecto en Windows)
df.to_csv('dataset_nlp_final_limpio.csv', index=False, encoding='utf-8-sig')

print("¡Ahora sí! Revisa el archivo 'dataset_nlp_final_limpio.csv'")

¡Ahora sí! Revisa el archivo 'dataset_nlp_final_limpio.csv'
